This script derives GeoCluster-level land-cover composition from the CORINE Land Cover 2018 raster for AI-relevant habitats. It first loads country polygons from the Natural Earth shapefile, reprojects them to the CLC CRS (EPSG:3035), filters to a predefined set of 38 European ISO3 codes, drops Montenegro, and merges in a cluster_lookup_auto.csv table to attach each country to a GeoCluster. It then uses rasterstats.zonal_stats (with categorical counts) to compute, for each country polygon, the number of 100 m × 100 m CLC grid cells (CELL_AREA_KM2 = 0.01 km²) in each land-cover class, caching a filtered version (only AI_CLC_CODES) in filtered_stats.pkl for speed. These raw counts are flattened into a long table with one row per ISO3 × cluster × CLC_code, and each CLC code is mapped to one of four broad land-cover categories (AgriculturalLand, ForestAreas, Wetlands, WaterBodies) via CAT_MAP. The script then aggregates counts to the GeoCluster × category level, converts counts to area in km², and computes within-cluster proportions (area of a category divided by total AI-relevant area in that cluster). It exports two summary CSVs—cluster_category_area.csv (cluster × category area_km2) and cluster_category_proportion.csv (cluster × category prop)—and prints wide tables to the console for quick inspection. Finally, for each of the four main categories, it writes two more TSV files (one with GeoCluster-specific areas, e.g. AgriculturalLand_areas.tsv, and one with proportions, e.g. AgriculturalLand_prop.tsv), producing eight GeoCluster-level land-cover covariate tables for downstream analyses.

In [2]:
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats
import pickle
from pathlib import Path

# ── user-defined paths / constants ──────────────────────────────────────────
ISO38 = [
    "ALB","AUT","BEL","BIH","BGR","CHE","CYP","CZE","DEU","DNK","ESP","EST",
    "FIN","FRA","GBR","GRC","HRV","HUN","IRL","ISL","ITA","XKX","LTU","LUX",
    "LVA","MDA","MKD","MNE","NLD","NOR","POL","PRT","ROU","SRB","SVK","SVN",
    "SWE","UKR"
]
CLC_RASTER = "/home/ss11645/Landcover/U2018_CLC2018_V2020_20u1.tif"
SHAPEFILE = "/home/ss11645/Landcover/shapes/ne_110m_admin_0_countries.shp"
LOOKUP_CSV = "cluster_lookup_auto.csv"  # uploaded file
CELL_AREA_KM2 = 0.01  # 100 m × 100 m
PICKLE_PATH = "filtered_stats.pkl"  # optional cache

# CLC codes relevant for avian-influenza analyses
AI_CLC_CODES = [
    10,12,13,14,18,21,22,23,24,25,27,28,29,30,31,32,35,36,37,38,39,40,41,42,43,44
]

# map each CLC code to a broad land-cover category
CAT_MAP = {
    # Agricultural land
    10:"AgriculturalLand",
    12:"AgriculturalLand",
    13:"AgriculturalLand",
    14:"AgriculturalLand",
    18:"AgriculturalLand",
    21:"AgriculturalLand",
    22:"AgriculturalLand",
    # Forest
    23:"ForestAreas",
    24:"ForestAreas",
    25:"ForestAreas",
    27:"ForestAreas",
    28:"ForestAreas",
    30:"ForestAreas",
    31:"ForestAreas",
    32:"ForestAreas",
    # Wetlands
    37:"Wetlands",
    38:"Wetlands",
    39:"Wetlands",
    # Water
    40:"WaterBodies",
    41:"WaterBodies",
    42:"WaterBodies",
    43:"WaterBodies",
    44:"WaterBodies"
}

# ─────────────────────────────────────────────────────────────────────────────
# 1. load polygons + merge GeoCluster labels
# ─────────────────────────────────────────────────────────────────────────────
countries = (
    gpd.read_file(SHAPEFILE)
      .set_crs(epsg=4326).to_crs(epsg=3035)  # match CLC raster CRS
      .loc[lambda d: d["ISO_A3_EH"].isin(ISO38), ["ISO_A3_EH", "ADMIN", "geometry"]]
      .rename(columns={"ISO_A3_EH": "ISO3", "ADMIN": "Country"})
      .reset_index(drop=True)
)
countries = countries[countries["Country"] != "Montenegro"]

cluster_lookup = (
    pd.read_csv(LOOKUP_CSV)
      .rename(columns=str.strip)  # strip errant spaces
      .rename(columns={"GeoCluster": "cluster"})
)
countries = countries.merge(cluster_lookup, on="Country", how="left")

if countries["cluster"].isna().any():
    missing = countries.loc[countries["cluster"].isna(), "Country"].tolist()
    raise ValueError(f"No GeoCluster found for: {', '.join(missing)}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. zonal-stats – use cache if available
# ─────────────────────────────────────────────────────────────────────────────
if Path(PICKLE_PATH).exists():
    with open(PICKLE_PATH, "rb") as f:
        filtered_stats = pickle.load(f)
else:
    raw_stats = zonal_stats(
        countries["geometry"],  # iterable of Shapely geometries
        CLC_RASTER,
        categorical=True,
        nodata=0
    )
    filtered_stats = [
        {code: cnt for code, cnt in d.items() if code in AI_CLC_CODES}
        for d in raw_stats
    ]
    with open(PICKLE_PATH, "wb") as f:
        pickle.dump(filtered_stats, f)


# ─────────────────────────────────────────────────────────────────────────────
# 3. long table: one row per ISO × CLC_code
# ─────────────────────────────────────────────────────────────────────────────
country_long = pd.DataFrame(
    [
        {"ISO3": iso, "cluster": clu, "CLC_code": code, "count": cnt}
        for iso, clu, d in zip(countries.ISO3, countries.cluster, filtered_stats)
        for code, cnt in d.items()
    ]
)
# add broad land-cover category
country_long["category"] = country_long["CLC_code"].map(CAT_MAP)

# ─────────────────────────────────────────────────────────────────────────────
# 4. aggregate counts to GeoCluster × category
# ─────────────────────────────────────────────────────────────────────────────
cluster_cat_cnt = (
    country_long
      .groupby(["cluster", "category"], as_index=False)["count"]
      .sum()
)
cluster_cat_cnt["area_km2"] = cluster_cat_cnt["count"] * CELL_AREA_KM2

# within-cluster proportions
totals = (
    cluster_cat_cnt.groupby("cluster", as_index=False)["count"]
      .sum()
      .rename(columns={"count": "total"})
)
cluster_cat_prop = (
    cluster_cat_cnt
      .merge(totals, on="cluster")
      .assign(prop=lambda d: d["count"] / d["total"])
      .drop(columns=["count", "total"])
)

# ─────────────────────────────────────────────────────────────────────────────
# 5. export results
# ─────────────────────────────────────────────────────────────────────────────
cluster_cat_cnt[["cluster", "category", "area_km2"]].to_csv(
    "cluster_category_area.csv", index=False
)
cluster_cat_prop[["cluster", "category", "prop"]].to_csv(
    "cluster_category_proportion.csv", index=False
)

print("✅ Saved:")
print(" • cluster_category_area.csv")
print(" • cluster_category_proportion.csv")

# optional: pretty print wide tables to screen
wide_area = (
    cluster_cat_cnt
      .pivot(index="cluster", columns="category", values="area_km2")
      .fillna(0)
      .sort_index()
)
wide_prop = (
    cluster_cat_prop
      .pivot(index="cluster", columns="category", values="prop")
      .fillna(0)
      .sort_index()
)

print("\nArea by land-cover category (km²):")
print(wide_area.round(1))

print("\nProportion of each GeoCluster occupied by category:")
print((wide_prop * 100).round(2).astype(str) + " %")

# ─────────────────────────────────────────────────────────────────────────────
# 6. write one TSV per category → 8 files total
# ─────────────────────────────────────────────────────────────────────────────
for cat in ["AgriculturalLand", "ForestAreas", "Wetlands", "WaterBodies"]:
    # 6-A. areas (km²) ────────────────────────────────────────────────────
    area_df = (
        cluster_cat_cnt.loc[cluster_cat_cnt["category"] == cat, ["cluster", "area_km2"]]
          .rename(columns={
              "cluster": "GeoCluster",
              "area_km2": f"{cat}_areas"
          })
          .sort_values("GeoCluster")
    )
    area_df.to_csv(f"{cat}_areas.tsv", sep="\t", index=False)

    # 6-B. proportions (0-to-1) ───────────────────────────────────────────
    prop_df = (
        cluster_cat_prop.loc[cluster_cat_prop["category"] == cat, ["cluster", "prop"]]
          .rename(columns={
              "cluster": "GeoCluster",
              "prop": f"{cat}_prop"
          })
          .sort_values("GeoCluster")
    )
    prop_df.to_csv(f"{cat}_prop.tsv", sep="\t", index=False)

print("✅ Extra TSV files saved:")
print(" • AgriculturalLand_areas.tsv • AgriculturalLand_prop.tsv")
print(" • ForestAreas_areas.tsv • ForestAreas_prop.tsv")
print(" • Wetlands_areas.tsv • Wetlands_prop.tsv")
print(" • WaterBodies_areas.tsv • WaterBodies_prop.tsv")


✅ Saved:
 • cluster_category_area.csv
 • cluster_category_proportion.csv

Area by land-cover category (km²):
category          AgriculturalLand  ForestAreas  WaterBodies  Wetlands
cluster                                                               
GeoCluster_Five           204680.2     235697.2      26431.1     403.0
GeoCluster_Four           141609.5     816690.9     137298.7     426.7
GeoCluster_One            250500.3     198761.6      29442.8     429.8
GeoCluster_Three          893619.1     621183.2     103980.3   11032.3
GeoCluster_Two            250777.6     134555.0       9840.5       6.4

Proportion of each GeoCluster occupied by category:
category         AgriculturalLand ForestAreas WaterBodies Wetlands
cluster                                                           
GeoCluster_Five           43.81 %     50.45 %      5.66 %   0.09 %
GeoCluster_Four           12.92 %     74.51 %     12.53 %   0.04 %
GeoCluster_One            52.28 %     41.48 %      6.14 %   0.09 %
GeoClu